# Headless Reduction Pipeline

`ReductionPlan` owns the scientific settings; `run_reduction` owns
execution; a sink owns output. This is the recommended batch-reduction
path and deliberately has no Qt dependency.


In [ ]:
import os
from IPython import get_ipython

# Equivalent to %matplotlib widget; headless checks explicitly use inline.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", os.environ.get("XDART_NOTEBOOK_BACKEND", "widget"))

import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display

from xrd_tools.core.containers import PONI
from xrd_tools.core.scan import Scan, ScanFrame
from xrd_tools.integrate import load_poni
from xrd_tools.io import read_image
from xrd_tools.reduction import CancelToken, Integration1DPlan, Integration2DPlan, MemorySink, ReductionPlan, run_reduction


In [ ]:
import os
from pathlib import Path

# Smoke mode is deterministic and has no external-data requirement.
# Set XDART_NOTEBOOK_SMOKE=0 and XDART_TEST_DATA=/path/to/data for real data.
SMOKE_MODE = os.environ.get("XDART_NOTEBOOK_SMOKE", "1") != "0"
TEST_DATA = Path(os.environ.get("XDART_TEST_DATA", "")).expanduser()


In [ ]:
image_paths = []  # Real mode: bounded ordered detector images.
poni_file = None  # Real mode: calibration for image_paths.
npt_1d = widgets.BoundedIntText(value=128, min=32, max=4096, description="1-D points")
include_2d = widgets.Checkbox(value=True, description="also integrate 2-D")
run_button = widgets.Button(description="Run reduction", button_style="primary")
cancel_button = widgets.Button(description="Cancel active run")
chunk_size = widgets.BoundedIntText(value=4, min=1, max=64, description="chunk")
progress = widgets.IntProgress(value=0, min=0, max=1, description="frames")
status = widgets.HTML("<i>Run creates a real MemorySink; cancel requests the next frame boundary.</i>")
output = widgets.Output()
display(widgets.VBox([widgets.HBox([npt_1d, chunk_size, include_2d]), widgets.HBox([run_button, cancel_button]), progress, status, output]))


In [ ]:
NOTEBOOK_STATE = {"runs": 0, "sink": None, "result": None, "cancel_token": None}

def _smoke_scan():
    shape = (195, 487)
    poni = PONI(dist=0.2, poni1=shape[0] * 172e-6 / 2, poni2=shape[1] * 172e-6 / 2,
                rot1=0.0, rot2=0.0, rot3=0.0, wavelength=1e-10, detector="Pilatus100k")
    yy, xx = np.mgrid[:shape[0], :shape[1]]
    radius = np.sqrt((yy - shape[0] / 2) ** 2 + (xx - shape[1] / 2) ** 2)
    image = 500 * np.exp(-((radius - 60) / 10) ** 2) + 3
    return Scan("synthetic_reduction", [ScanFrame(index, image=image * (1 + 0.05 * index)) for index in range(2)], poni=poni)

def _real_scan():
    assert image_paths and poni_file is not None, "Configure image_paths and poni_file for real reduction"
    paths = sorted(map(Path, image_paths), key=lambda path: path.name)
    return Scan("configured_reduction", [ScanFrame(index, image=read_image(path), source_path=path) for index, path in enumerate(paths)], poni=load_poni(poni_file))

def cancel_active_run(_=None):
    token = NOTEBOOK_STATE.get("cancel_token")
    if token is not None:
        token.cancel()
        status.value = "<b>Cancellation requested at the next frame boundary.</b>"

def run_pipeline(_=None):
    with output:
        clear_output(wait=True)
        try:
            scan = _smoke_scan() if SMOKE_MODE else _real_scan()
            plan = ReductionPlan(integration_1d=Integration1DPlan(npt=npt_1d.value), integration_2d=Integration2DPlan(npt_rad=max(32, npt_1d.value // 2), npt_azim=32) if include_2d.value else None)
            sink, token = MemorySink(), CancelToken()
            NOTEBOOK_STATE["cancel_token"] = token
            progress.max, progress.value = len(scan), 0
            def update(event):
                progress.value = min(event.completed, progress.max)
            result = run_reduction(plan, scan, sink=sink, chunk_size=chunk_size.value, progress_cb=update, cancel_token=token)
            assert sink.frames and result.n_processed == len(sink.frames)
            display({"processed": result.n_processed, "cancelled": result.cancelled, "sink_frames": sorted(sink.frames)})
            NOTEBOOK_STATE.update(runs=NOTEBOOK_STATE["runs"] + 1, sink=sink, result=result)
            status.value = f"<b>Reduced {result.n_processed} frames into MemorySink.</b>"
        except Exception as exc:
            status.value = f"<b>Reduction failed:</b> {exc}"
            raise
        finally:
            NOTEBOOK_STATE["cancel_token"] = None

run_button.on_click(run_pipeline)
cancel_button.on_click(cancel_active_run)
NOTEBOOK_ACTIONS = {"run_reduction": run_pipeline, "cancel_reduction": cancel_active_run}
if SMOKE_MODE or os.environ.get("XDART_NOTEBOOK_AUTORUN") == "1":
    run_pipeline()
